# SoundStream training on Kaggle

**Before running:**
1. Add dataset [LibriSpeech](https://www.kaggle.com/datasets/a24998667/librispeech) (or update `LIBRI_INPUT` below).
2. Add Kaggle secrets:
   - `GITHUB_TOKEN` - if the repo is private
   - `WANDB_API_KEY` - for W&B logging
3. Enable GPU.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import shutil
import time
import torch

REPO_DIR = "soundstream_hw"
REPO_URL = "https://github.com/ndrew1337/soundstream_hw.git"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = f"https://{token}@github.com/ndrew1337/soundstream_hw.git"
except Exception:
    pass

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
if str(Path(REPO_DIR).resolve()) not in sys.path:
    sys.path.insert(0, str(Path(REPO_DIR).resolve()))

In [ ]:
os.environ["PIP_NO_WARN_CONFLICTS"] = "1"
!pip install -q torchmetrics pystoi "numba>=0.59" librosa soundfile requests tqdm wget matplotlib pandas wandb hydra-core omegaconf

In [ ]:
LIBRI_INPUT = "/kaggle/input/datasets/a24998667/librispeech"

os.makedirs("/kaggle/working/lsdata", exist_ok=True)
for part in ["train-clean-100", "test-clean"]:
    part_src = f"{LIBRI_INPUT}/{part}"
    part_dst = f"/kaggle/working/lsdata/{part}"
    if not os.path.exists(part_dst):
        os.symlink(part_src, part_dst)
!ls /kaggle/working/lsdata/

In [ ]:
N_EPOCHS = 120
RUN_NAME = f"v5-kaggle-{N_EPOCHS}ep"
CKPT = Path(f"saved/{RUN_NAME}") / f"checkpoint-epoch{N_EPOCHS}.pth"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    pass

import wandb
wandb.login()

print(f"Epochs: {N_EPOCHS}")
print(f"Run: {RUN_NAME}")
print(f"Checkpoint: {CKPT}")

## Train

In [ ]:
t0 = time.time()
!HYDRA_FULL_ERROR=1 python train.py -cn baseline_kaggle \
    datasets.train.data_dir=/kaggle/working/lsdata \
    datasets.test.data_dir=/kaggle/working/lsdata \
    trainer.n_epochs={N_EPOCHS} \
    writer.run_name={RUN_NAME} \
    writer.mode=online \
    trainer.override=True
print(f"Elapsed: {(time.time() - t0) / 3600:.2f} h")

## Inference 

In [ ]:
if not Path(CKPT).is_file():
    raise FileNotFoundError(f"Train first or set CKPT. Missing: {CKPT}")

!HYDRA_FULL_ERROR=1 python inference.py \
    datasets.test.data_dir=/kaggle/working/lsdata \
    inferencer.from_pretrained={CKPT} \
    inferencer.save_path=test-clean-final \
    inferencer.save_audio=false \
    inferencer.save_visuals_count=0

In [ ]:
ckpt_src = Path(CKPT)
if not ckpt_src.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_src.resolve()}")

out_dir = Path("/kaggle/working/output")
out_dir.mkdir(parents=True, exist_ok=True)
ckpt_dst = out_dir / f"checkpoint-epoch{N_EPOCHS}.pth"
shutil.copy2(ckpt_src, ckpt_dst)
print(f"Saved to {ckpt_dst} ({ckpt_dst.stat().st_size / 1e6:.1f} MB)")